# Iris Dataset Analysis

A complete exploratory data analysis of the classic Iris dataset — running entirely on-device via JupyterDroid.

**What we'll cover:**
1. Load and inspect the data
2. Descriptive statistics
3. Per-species breakdown
4. Feature distributions (ASCII histograms)
5. Correlation matrix
6. K-Nearest Neighbors classification

> **Before running:** use the pip install button (bottom toolbar) to install `numpy pandas scikit-learn`

In [ ]:
import numpy as np
import pandas as pd
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

print(f"numpy  {np.__version__}")
print(f"pandas {pd.__version__}")
print("scikit-learn imported")
print("Ready.")

## 1. Load the Data

In [ ]:
iris = datasets.load_iris()

df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['species'] = pd.Series(iris.target).map(
    {0: 'setosa', 1: 'versicolor', 2: 'virginica'}
)

print(f"Shape : {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Features: {iris.feature_names}")
print()
print("Samples per species:")
print(df['species'].value_counts().to_string())

## 2. First Look

In [ ]:
print("First 10 rows:")
print("-" * 72)
print(df.head(10).to_string(index=True))

## 3. Descriptive Statistics

In [ ]:
print("Overall descriptive statistics (all species combined):")
print("=" * 72)
print(df.drop('species', axis=1).describe().round(2).to_string())

## 4. Per-Species Breakdown

In [ ]:
features = [c for c in df.columns if c != 'species']

print("Mean by species:")
print("=" * 72)
print(df.groupby('species')[features].mean().round(3).to_string())

print()
print("Std deviation by species:")
print("=" * 72)
print(df.groupby('species')[features].std().round(3).to_string())

print()
print("Min / Max by species:")
print("=" * 72)
print(df.groupby('species')[features].agg(['min', 'max']).round(2).to_string())

## 5. Feature Distributions (ASCII Histograms)

In [ ]:
def ascii_hist(values, title, bins=10, width=36):
    lo, hi = values.min(), values.max()
    step = (hi - lo) / bins
    counts = [0] * bins
    for v in values:
        idx = min(int((v - lo) / step), bins - 1)
        counts[idx] += 1
    peak = max(counts)
    print(f"\n  {title}")
    print("  " + "-" * (width + 18))
    for i, n in enumerate(counts):
        a = lo + i * step
        b = a + step
        bar = "█" * int(n / peak * width)
        print(f"  {a:4.1f}–{b:4.1f} │{bar:<{width}}│ {n:3}")
    print()

for col in features:
    ascii_hist(df[col], col)

## 6. Correlation Matrix

In [ ]:
corr = df[features].corr().round(3)

print("Pearson Correlation Matrix:")
print("=" * 72)
print(corr.to_string())

print()
print("Strongest correlations:")
pairs = []
for i in range(len(features)):
    for j in range(i + 1, len(features)):
        pairs.append((abs(corr.iloc[i, j]), features[i], features[j], corr.iloc[i, j]))
for _, a, b, r in sorted(pairs, reverse=True):
    direction = "positive" if r > 0 else "negative"
    print(f"  {a:30s} ↔ {b:30s}  r = {r:+.3f}  ({direction})")

## 7. Classification: K-Nearest Neighbors

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.3, random_state=42
)

clf = KNeighborsClassifier(n_neighbors=3)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)
names = iris.target_names

print(f"K-Nearest Neighbors (k=3)")
print(f"Train: {len(X_train)} samples  |  Test: {len(X_test)} samples")
print(f"Accuracy: {acc:.1%}  ({int(acc * len(y_test))}/{len(y_test)} correct)")

print()
print("Confusion Matrix (rows = actual, cols = predicted):")
print()
header = f"{'':18}" + "".join(f"{n:>12}" for n in names)
print(header)
print("  " + "-" * (len(header) - 2))
for i, row in enumerate(cm):
    cells = "".join(f"{v:>12}" for v in row)
    print(f"  actual {names[i]:<10}{cells}")

print()
misclassified = [(names[y_test[i]], names[y_pred[i]])
                 for i in range(len(y_test)) if y_test[i] != y_pred[i]]
if misclassified:
    print(f"Misclassified ({len(misclassified)}):")
    for actual, predicted in misclassified:
        print(f"  actual={actual}  predicted={predicted}")
else:
    print("No misclassifications on the test set.")

## 8. Summary

**Key findings:**

- The Iris dataset has 150 samples equally split across 3 species.
- *Setosa* is clearly separable from the other two species — its petal length and width are much smaller.
- *Versicolor* and *Virginica* overlap slightly, making them harder to distinguish.
- Petal length and petal width are strongly correlated (r ≈ 0.96) and are the most discriminative features.
- A simple K-Nearest Neighbors classifier (k=3) achieves ~97% accuracy on a 70/30 train-test split.

All of this ran locally on your Android device — no internet, no server.